# TTA Param Sweep

Notebook này sweep param cho `none`, `tip_adapter`, `freetta`, `bca`.

Mặc định chạy nhẹ trước trên:

- `ffpp-test`
- `ffpp-test-corruption-balanced`

với model `linear-probe`. Sau khi chọn param tốt có thể mở rộng sang model/dataset khác.

## Kaggle Setup

In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
!pip install -q -e . --no-deps

## Check Mounted Inputs

In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 3 -type f \( -name "*.pt" -o -name "*.csv" \) | sort | sed -n "1,160p"

## Run Compact Sweep

Grid mặc định:

- Tip-Adapter: shots `[8, 16, 32]`, alpha `[0.2, 0.5, 0.8]`, beta `[3.5, 5.5, 7.5]`
- FreeTTA: base weight `[0.2, 0.4, 0.6]`, momentum `[0.9, 0.95, 0.98]`
- BCA: temperature `[0.03, 0.07, 0.12]`, base weight `[0.3, 0.5, 0.7]`


In [ ]:
FFPP_SPLIT = "/kaggle/input/ffpp-split-features"
FFPP_CORR = "/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings"
MODELS = "/kaggle/input/ffpp-training-free-models"

!python testing/evaluate_tta_param_sweep.py \
  --train-features {FFPP_SPLIT}/ffpp_train_features.pt \
  --dataset ffpp-test={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption-balanced={FFPP_CORR} \
  --balanced-aligned-dataset ffpp-test-corruption-balanced \
  --model linear-probe={MODELS}/ffpp_linear_probe_split.pt \
  --thresholds-csv linear-probe={MODELS}/ffpp_probe_thresholds.csv \
  --methods none tip_adapter freetta bca \
  --tip-shots 16 \
  --tip-alpha 0.2 0.5 0.8 \
  --tip-beta 3.5 5.5 7.5 \
  --freetta-base-weight 0.2 0.4 0.6 \
  --freetta-momentum 0.9 0.95 0.98 \
  --freetta-prior-power 1.0 \
  --bca-temperature 0.03 0.07 0.12 \
  --bca-base-weight 0.3 0.5 0.7 \
  --bca-prior-momentum 0.95 \
  --bca-prototype-momentum 0.98 \
  --results-output /kaggle/working/tta_param_sweep_results.csv \
  --eval-batch-size 4096 \
  --test-batch-size 512 \
  --cache-batch-size 8192 \
  --block-size 10 \
  --shuffle-all-tests \
  --device auto \
  --continue-on-error

## Preview Results

In [ ]:
import pandas as pd

results = pd.read_csv("/kaggle/working/tta_param_sweep_results.csv")
display(results.head())
display(results.sort_values(["dataset", "method", "f1"], ascending=[True, True, False]).head(30))

## Best Params

In [ ]:
metric = "f1"
valid = results[results["error"].isna()] if "error" in results.columns else results
summary = valid.groupby(["dataset", "method", "param_id"], dropna=False)[["acc", "f1", "auc", "ap", "eer"]].mean().reset_index()
best = summary.sort_values(["dataset", "method", metric], ascending=[True, True, False]).groupby(["dataset", "method"], as_index=False).head(3)
summary.to_csv("/kaggle/working/tta_param_sweep_summary.csv", index=False)
best.to_csv("/kaggle/working/tta_param_sweep_best.csv", index=False)
display(best)

## Wider Model Sweep

Sau khi thấy grid ổn, có thể thêm model balanced/OSD bằng cách thêm các dòng `--model ...` vào command Run.